In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device using:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

In [ ]:
from torchvision import transforms
from src.dataset import mask_to_long_tensor # needs to be here because slow otherwise

# SPECS1 =  CPU 10700KF, RAM 32GB, GPU RTX 2080S 8GB VRAM

#Original = (1024, 2048) # 2,097,152
#IMG_SIZE = (512, 1024)  # 524,288
IMG_SIZE = (256, 512)   # 131,072 # OK using SPECS1 (8 batch, 12 workers) 20 epoch = ~13 mins (~6GB VRAM used)
#IMG_SIZE = (128, 256)   # 32,768 # EASY using SPECS1 (8 batch, 12 workers) 20 epoch = ~5 mins (~2GB VRAM used)
#IMG_SIZE = (64, 128)    # 8,192

# Images are continuous colour gradients, while the masks are categorical labels. Use different interpolation methods for each.
# Docs: https://docs.pytorch.org/vision/0.9/_modules/torchvision/transforms/functional.html#InterpolationMode
image_transform = transforms.Compose([
    # BILINEAR averages neighboring pixels to produce smooth gradients
    transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor() # tensor scaled to [0, 1]
])

target_transform = transforms.Compose([
    # NEAREST preserves original values
    transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(), # tensor without scaling (stays 0-255)
    # remove dummy channel dimension and cast to int64 because PyTorch's CrossEntropyLoss requires this for segmentation. [1, H, W] -> [H, W]
    transforms.Lambda(mask_to_long_tensor),
])

In [ ]:
import torch
import os
from torch.utils.data import DataLoader
from src.dataset import CityscapesDataset

train_ds = CityscapesDataset("./data/dataset/train", image_transform, target_transform)
valid_ds = CityscapesDataset("./data/dataset/val", image_transform, target_transform)
test_ds = CityscapesDataset("./data/dataset/test", image_transform, target_transform)

BATCH_SIZE = 8
NUM_WORKERS = min(12, os.cpu_count() or 1)
print("Dataloader workers:", NUM_WORKERS)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True, # useful for stochastic gradient descent
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False, # evaluate samples in same order
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

test_loader = DataLoader(test_ds, batch_size=4, shuffle=True, num_workers=0)

In [ ]:
from src.model import UNet

In [ ]:
model = UNet().to(device)

imgs, masks = next(iter(train_loader))
print("images:", imgs.shape, imgs.dtype)    # [B,3,H,W] float32
print("masks :", masks.shape, masks.dtype)  # [B,H,W]   int64
print("mask min/max:", masks.min().item(), masks.max().item())  # 0..18 and/or 255

with torch.no_grad():
    out = model(imgs.to(next(model.parameters()).device))
print("logits:", out.shape)  # [B,19,H,W]

In [ ]:
model = UNet(in_channels=3, num_classes=19, base=32).to(device)

In [ ]:
import torch
import torch.nn as nn
from tqdm import tqdm

criterion = nn.CrossEntropyLoss(ignore_index=255) # ignore unlabelled
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

def run_epoch(loader, train: bool):
    model.train(train)
    total_loss = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, masks in tqdm(loader, leave=False):
            imgs  = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).long()

            logits = model(imgs)
            loss = criterion(logits, masks)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
    return total_loss / len(loader)

EPOCHS = 20
best_val = float("inf")

for epoch in range(EPOCHS):
    tr = run_epoch(train_loader, True)
    va = run_epoch(valid_loader, False)
    print(f"epoch {epoch:02d}  train {tr:.4f}  val {va:.4f}")

    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), "./models/best_unet.pt")

In [ ]:
model.load_state_dict(torch.load("./models/best_unet.pt"))
model.eval()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from cityscapesscripts.helpers.labels import labels

# Official colours and names for trainIds
# Source: https://github.com/mcordts/cityscapesScripts/blob/master/cityscapesscripts/helpers/labels.py

# keep only trainIds (0-18), sort by trainId
classes = sorted(
    [l for l in labels if 0 <= l.trainId < 19],
    key=lambda l: l.trainId
)

# matplotlib expects float RGB values [0, 1], so divide by 255 to convert
colors = np.array([l.color for l in classes]) / 255
# create colour map with 19 colours + black for unlabelled
cmap = ListedColormap(colors)
cmap.set_over("black")

# get a test image and prediction
imgs, masks = next(iter(test_loader))
model.eval()
with torch.no_grad():
    pred = model(imgs.to(device)).argmax(1).cpu()

# create plot
fig, ax = plt.subplots(3, 1, figsize=(12, 12))

# matplotlib expects (height, width, 3) instead of (3, height, width) so rearrange
ax[0].imshow(imgs[0].permute(1, 2, 0).cpu())
ax[0].set_title("Real Image")

ax[1].imshow(masks[0].cpu(), cmap=cmap, vmin=0, vmax=18)
ax[1].set_title("True Mask")

ax[2].imshow(pred[0], cmap=cmap, vmin=0, vmax=18)
ax[2].set_title("Predicted Mask")

# hide plots axis, numbers, borders
for a in ax:
    a.axis("off")

# colour and name + add black for unlabelled
handles = [Patch(color=colors[i], label=l.name) for i, l in enumerate(classes)]
handles.append(Patch(color="black", label="unlabelled"))

fig.legend(handles=handles, loc="center left", bbox_to_anchor=(0.7, 0.5), title="Classes")

plt.tight_layout(rect=[0, 0, 0.8, 1])
plt.show()